# Redrob Semantic Ranker — Phase 2

Combines, for each candidate:
- **structured fit** (`struct_score` from explore.ipynb: product-company ML, retrieval/eval evidence, verified IR assessment, location, experience band; positive fit ignores noisy skill-list keywords)
- **semantic fit** = cosine(JD, candidate **career prose**) via `bge-small` — catches the no-buzzword Tier-5 and the meaning the phrase-lists miss
- **behavioral** response/recency multiplier (JD directive #3)
- **logistics** India eligibility, relocation, and notice multiplier; abroad candidates are heavily suppressed because sponsorship is unavailable

`final = (0.75 * struct_norm + 0.25 * sem_norm) * behavioral * logistics`, honeypots excluded.
Candidate and JD embeddings are precomputed once and cached (the 5-min live ranker just loads arrays).

In [1]:
import json, os, time
import numpy as np, pandas as pd

DATA='data/candidates.jsonl'
MAX_SEQ_LENGTH = 256
EMB=f'artifacts/cand_emb_{MAX_SEQ_LENGTH}.npy'
IDS='artifacts/cand_ids.json'
JD_EMB=f'artifacts/jd_emb_{MAX_SEQ_LENGTH}.npy'

feat=pd.read_parquet('artifacts/features.parquet').set_index('candidate_id')
print('features:', feat.shape, '| honeypots:', int(feat.honeypot.sum()))

# JD distilled to the fit-theory, used as the retrieval query (bge wants the query prefix).
# Cache this too so the final ranking step can run with no network/model call.
JD_QUERY=('Represent this sentence for searching relevant passages: '
  'Senior AI/ML engineer who has built and shipped ranking, retrieval, search, recommendation and '
  'matching systems to real users in production at product companies. Embeddings-based retrieval, '
  'vector search, hybrid search, learning-to-rank, semantic search; rigorous evaluation with NDCG, MRR, '
  'MAP, A/B testing and offline-online correlation. Strong Python. 5-9 years experience, applied ML at '
  'product companies rather than pure services. Based in or willing to relocate to Pune or Noida, India.')

model = None
if os.path.exists(JD_EMB):
    jd_emb=np.load(JD_EMB)
    print('loaded cached JD embedding:', jd_emb.shape)
else:
    import torch
    from sentence_transformers import SentenceTransformer
    # Use Apple MPS / CUDA only for one-time precomputation; cached ranking is CPU/no-network.
    DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
    model = SentenceTransformer('BAAI/bge-small-en-v1.5', device=DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH
    print('device:', model.device, '| max_seq_length:', model.max_seq_length)
    jd_emb=model.encode(JD_QUERY, normalize_embeddings=True)
    np.save(JD_EMB, jd_emb)
    print('cached JD embedding:', jd_emb.shape)


features: (100000, 44) | honeypots: 80
loaded cached JD embedding: (384,)


In [2]:
# Build candidate 'work text' = summary + job titles + job descriptions (NOT the noise skill list),
# encode with bge-small, and CACHE. Re-runs load the cache instantly.
# Cache key includes max_seq_length so changing it forces a clean re-encode.
if os.path.exists(EMB) and os.path.exists(IDS):
    cand_ids=json.load(open(IDS)); cand_emb=np.load(EMB)
    print('loaded cached embeddings:', cand_emb.shape)
else:
    if model is None:
        import torch
        from sentence_transformers import SentenceTransformer
        DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
        model = SentenceTransformer('BAAI/bge-small-en-v1.5', device=DEVICE)
        model.max_seq_length = MAX_SEQ_LENGTH
        print('device:', model.device, '| max_seq_length:', model.max_seq_length)
    cand_ids=[]; texts=[]
    with open(DATA) as f:
        for line in f:
            line=line.strip()
            if not line: continue
            c=json.loads(line); p=c.get('profile',{}) or {}
            parts=[p.get('summary') or '']
            for j in (c.get('career_history',[]) or []):
                parts.append(j.get('title') or ''); parts.append(j.get('description') or '')
            cand_ids.append(c['candidate_id']); texts.append('. '.join(t for t in parts if t))
    bs = 256 if model.device.type in ('mps','cuda') else 64   # smaller batches are faster on CPU
    print(f'encoding {len(texts)} candidates on {model.device} (one-time)...')
    t=time.time()
    cand_emb=model.encode(texts, normalize_embeddings=True, batch_size=bs, show_progress_bar=True)
    print('encoded in', round(time.time()-t), 's')
    np.save(EMB, cand_emb); json.dump(cand_ids, open(IDS,'w'))
    print('cached:', cand_emb.shape)


loaded cached embeddings: (100000, 384)


In [ ]:
# Semantic similarity to JD, then the hybrid blend.
# We encoded the FULL pool (needed later for semantic-incoherence hunting of undetected honeypots),
# but we DROP the known honeypots up front so they can't contaminate normalization or the ranking.
# NB: column is 'sem_sim' not 'sem' -- 'sem' collides with the pandas DataFrame.sem() method.
sem_vec = cand_emb @ jd_emb                  # cosine (both normalized)
sem_df = pd.DataFrame({'candidate_id':cand_ids,'sem_sim':sem_vec}).set_index('candidate_id')

R_all = feat[~feat.honeypot].join(sem_df)        # genuine pool only (known honeypots excluded here)
# Normalize on the full genuine pool, then filter. This keeps semantic/structured calibration stable
# while preserving the final shortlist for the strongest JD-fit tier.
R_all['struct_norm'] = (R_all['struct_score'] / R_all['struct_score'].max()).clip(0,1)
lo,hi = R_all['sem_sim'].quantile(0.01), R_all['sem_sim'].quantile(0.99)
R_all['sem_norm'] = ((R_all['sem_sim'] - lo)/(hi-lo)).clip(0,1)

# ---- Skill-assessment: 40% pass floor + struct-based imputation for MISSING scores ----------
# The JD never requires an assessment, so MISSING is imputed from struct_score (our aggregate of
# experience + projects/evidence + skills + location) rather than penalized blindly. For the 4.5+ pool:
#   missing & struct >= struct_median  -> impute pool assessment median (~61)  "great"
#   missing & 6.5 <= struct < median   -> impute 40 (pass mark)                "fine"
#   missing & struct < 6.5             -> impute 0                             "doesn't meet"
# Then ANY 4.5+ candidate whose EFFECTIVE assessment < 40 is excluded (the 40% pass mark) --
# this drops both took-it-&-scored-<40 and missing-&-weak-struct candidates.
# (The 4.0-4.5 band keeps its own stricter gate below: floor 50, full 65, missing not imputed.)
ASSESS_PASS = 40.0
STRUCT_WEAK_FLOOR = 6.5
_mainpool = R_all[(R_all.tier==5) & (R_all.yoe>=4.5)]
ASSESS_MEDIAN = round(float(_mainpool['assess_avg'].median()), 1)    # ~61.1 among those who took it
STRUCT_MEDIAN = round(float(_mainpool['struct_score'].median()), 2)  # ~7.34
_s = R_all['struct_score']; _a = R_all['assess_avg']
_imputed = np.where(R_all['yoe'] < 4.5, 0.0,
             np.where(_s >= STRUCT_MEDIAN, ASSESS_MEDIAN,
               np.where(_s >= STRUCT_WEAK_FLOOR, ASSESS_PASS, 0.0)))
R_all['assess_eff'] = np.where(_a.notna(), _a, _imputed)   # taken score, else struct-imputed
print(f'assessment: pool median(taken)={ASSESS_MEDIAN}, struct median={STRUCT_MEDIAN}, pass floor={ASSESS_PASS}')

# --- Early-career (4.0-4.5 yoe) HARD assessment gate (floor 50, full 65; missing -> 0/excluded) ---
EARLY_FULL_ASSESS  = 65.0
EARLY_FLOOR_ASSESS = 50.0
EARLY_PARTIAL_MULT = 0.8
assess0 = R_all['assess_avg'].fillna(0.0)        # missing skill-assessment -> 0.0 for the early band
early_band = (R_all['yoe'] >= 4) & (R_all['yoe'] < 4.5)
early_exceptional = (
    early_band &
    (R_all['retrieval_evidence'] >= 2) &
    (R_all['prod_evidence'] >= 2) &
    ((R_all['eval_evidence'] >= 1) | (R_all['vector_infra_evidence'] >= 2) | (R_all['ir_assess'] >= 50)) &
    (assess0 >= EARLY_FLOOR_ASSESS)              # HARD assessment criterion for the 4.0-4.5 band
)
yoe_ok = (R_all['yoe'] >= 4.5) | early_exceptional
# 4.5+ must also clear the 40% effective-assessment floor; the early band is governed by its own gate.
pass_floor = (R_all['yoe'] < 4.5) | (R_all['assess_eff'] >= ASSESS_PASS)
eligible = yoe_ok & pass_floor

n_band = int(early_band.sum())
n_band_excl = int((early_band & (assess0 < EARLY_FLOOR_ASSESS)).sum())
_main = (R_all.tier==5) & (R_all.yoe>=4.5)
n_main_excl = int((_main & (R_all['assess_eff'] < ASSESS_PASS)).sum())
n_taken_fail = int((_main & R_all['assess_avg'].notna() & (R_all['assess_avg'] < ASSESS_PASS)).sum())
n_miss_weak  = int((_main & R_all['assess_avg'].isna()  & (R_all['assess_eff'] < ASSESS_PASS)).sum())

R = R_all[(R_all.tier == 5) & eligible].copy()
if len(R) < 100:
    print('WARNING: fewer than 100 tier-5 candidates after gates; falling back to tier-5 passing YOE gate only')
    R = R_all[(R_all.tier==5) & yoe_ok].copy()
print('ranking pool (honeypots removed, tier-5 first):', len(R), 'of', len(feat))
print(f'4.0-4.5 band: {n_band} | excluded by assessment < {EARLY_FLOOR_ASSESS} (incl. missing): {n_band_excl}')
print(f'4.5+ excluded by 40% floor: {n_main_excl}  (took & <40: {n_taken_fail} | missing & weak-struct: {n_miss_weak})')
print('abroad candidates in ranking pool:', int((~R['india']).sum()), '| with prior India signal:', int(((~R['india']) & R.get('prior_india_work_signal', False)).sum()))

def notice_factor(days):
    # JD: sub-30 is preferred; 30+ stays in scope but must clear a higher bar.
    if pd.isna(days): return 0.40
    d=float(days)
    if d<=15: return 1.00
    if d<=30: return 0.70
    if d<=60: return 0.40
    if d<90: return 0.10
    return 0.00

def location_factor(r):
    # Sponsorship is unavailable: current India-based candidates are strongly preferred.
    # Abroad candidates get only a tiny fallback if career history hints prior India work;
    # willingness to relocate alone is not sufficient.
    if bool(r.get('india')):
        if bool(r.get('jd_city')):
            return 1.00
        if bool(r.get('willing_relocate')):
            return 0.94
        return 0.78
    if bool(r.get('prior_india_work_signal')) and bool(r.get('willing_relocate')):
        return 0.10
    return -0.25

def logistics_multiplier(r):
    return round(location_factor(r) * notice_factor(r.get('notice_days')), 3)

R['logistics'] = R.apply(logistics_multiplier, axis=1)

# Partial weightage for 4.0-4.5 candidates whose assessment is in [50,65); full (1.0) otherwise.
# (Candidates below 50 / missing were already excluded by the yoe_ok gate above.)
assess0_R = R['assess_avg'].fillna(0.0)
early_band_R = (R['yoe'] >= 4) & (R['yoe'] < 4.5)
R['early_band_mult'] = np.where(early_band_R & (assess0_R < EARLY_FULL_ASSESS), EARLY_PARTIAL_MULT, 1.0)
print('4.0-4.5 candidates on PARTIAL weightage (x%.2f):' % EARLY_PARTIAL_MULT, int((R['early_band_mult'] < 1.0).sum()))

# Structured criteria carry most of the weight because the JD has hard must-have requirements.
# Semantic similarity still helps separate profiles that use different wording for the same work.
R['fit']   = 0.75*R['struct_norm'] + 0.25*R['sem_norm']
R['base_final'] = R['fit'] * R['behavioral'] * R['logistics'] * R['early_band_mult'] # logistics + early-band gate move the shortlist

def assessment_refine(score):
    # Small positive-only second-stage signal. Operates on the EFFECTIVE (taken-or-imputed) score.
    if pd.isna(score): return 0.0
    s=float(score)
    if s > 70: return 0.30
    if s >= 50: return 0.20
    if s >= 30: return 0.10
    return 0.0

R['assessment_refine'] = R['assess_eff'].apply(assessment_refine)   # uses imputed score for missing
shortlist = R.sort_values(['base_final','candidate_id'], ascending=[False,True]).head(150).copy()
base_lo, base_hi = shortlist['base_final'].min(), shortlist['base_final'].max()
shortlist['base_norm_150'] = ((shortlist['base_final'] - base_lo) / (base_hi - base_lo)).clip(0,1) if base_hi > base_lo else 1.0
shortlist['final'] = 0.95*shortlist['base_norm_150'] + 0.05*shortlist['assessment_refine']

top = shortlist.sort_values(['final','base_final','candidate_id'], ascending=[False,False,True]).head(100)
print('top-150 base shortlist:', len(shortlist), '| final top-100 after assessment refinement:', len(top))
print('TOP-100 tier distribution:', top['tier'].value_counts().sort_index().to_dict())
print('final score range in top-100:', round(top['final'].min(),3),'..',round(top['final'].max(),3))
print('notice distribution:', top['notice_days'].value_counts().sort_index().to_dict())
print('outside India in top-100:', int((~top['india']).sum()))
if (~top['india']).any():
    print('abroad top-100 prior India signal:', int(((~top['india']) & top.get('prior_india_work_signal', False)).sum()))
debug_cols=['tier','struct_score','sem_sim','fit','behavioral','logistics','early_band_mult','base_final','assessment_refine','final','current_title','current_industry','yoe','profile_yoe','career_yoe','span_yoe','yoe_gap_span','profile_yoe_gt_span','country','location','prior_india_work_signal','willing_relocate','notice_days','retrieval_evidence','vector_infra_evidence','prod_evidence','eval_evidence','python_evidence','nice_to_have_hits','completeness','assess_avg','assess_eff','assess_max']
debug_cols=[c for c in debug_cols if c in top.columns]
top[debug_cols].head(20)


## Audit the top-100 for honeypots

The plan: rank first, then scrutinize only the shortlist. Re-run the full impossibility battery on the
top-100 raw records (should be 0, since honeypots are excluded), and surface them for manual inspection
of any semantic-incoherence honeypots the rules can't catch.

In [4]:
top_ids=set(top.index)
recs={}
with open(DATA) as f:
    for line in f:
        line=line.strip()
        if not line: continue
        c=json.loads(line)
        if c['candidate_id'] in top_ids: recs[c['candidate_id']]=c
known_hp = top.index.isin(feat[feat.honeypot].index)
print('known honeypots in top-100:', int(known_hp.sum()), '(should be 0)')
print()
print('Top-15 shortlist (for eyeballing fit + any incoherence):')
order=list(top.index)
for i,cid in enumerate(order[:15], start=1):
    p=recs[cid]['profile']; r=top.loc[cid]
    print()
    print(f"#{i} {cid} | {p.get('current_title')} @ {p.get('current_company')} ({p.get('current_industry')}) | effective_yoe={r['yoe']} | final={r['final']:.3f} sem={r['sem_sim']:.3f} tier={int(r['tier'])}")
    print('   ', (p.get('summary') or '')[:240])

known honeypots in top-100: 0 (should be 0)

Top-15 shortlist (for eyeballing fit + any incoherence):

#1 CAND_0066999 | Recommendation Systems Engineer @ Microsoft (Software) | effective_yoe=5.9 | final=0.960 sem=0.858 tier=5
    Machine learning engineer with 5.9 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment

#2 CAND_0027691 | NLP Engineer @ Haptik (Conversational AI) | effective_yoe=6.5 | final=0.942 sem=0.811 tier=5
    Machine learning engineer with 6.5 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment

#3 CAND_0045250 | Applied ML Engineer @ Rephrase.ai (AI/ML) | effective_yoe=6.6 | final=0.925 sem=0.858 tier=5
    Machine learning engineer with 6.6 years of experience 

In [5]:
# ===== EXPORT the top-100 (rank order) with scores + full records for inspection =====
import csv, datetime as dt, re

TODAY = dt.date.today()
def parse_date(s):
    try: return dt.date.fromisoformat(s)
    except Exception: return None

def evidence_terms(rec):
    text = ' '.join([
        rec.get('profile',{}).get('summary',''),
        ' '.join((j.get('title','')+' '+j.get('description','')) for j in rec.get('career_history',[]))
    ]).lower()
    term_map = [
        ('hybrid retrieval','hybrid retrieval'), ('semantic search','semantic search'),
        ('candidate-jd','candidate-JD matching'), ('recommendation system','recommendation systems'),
        ('ranking model','ranking models'), ('ranking system','ranking systems'),
        ('learning-to-rank','learning-to-rank'), ('bm25','BM25'), ('faiss','FAISS'),
        ('qdrant','Qdrant'), ('milvus','Milvus'), ('weaviate','Weaviate'), ('pinecone','Pinecone'),
        ('elasticsearch','Elasticsearch'), ('vector','vector search'), ('embedding','embedding-based retrieval'),
        ('ndcg','NDCG'), ('mrr','MRR'), ('a/b test','A/B testing'), ('offline','offline evaluation'),
        ('production','production deployment'), ('serving','serving real users'), ('latency','latency work'),
    ]
    out=[]
    for needle,label in term_map:
        if needle in text and label not in out:
            out.append(label)
    return out[:5]

def make_reasoning(rec, row):
    p=rec.get('profile',{}) or {}; sig=rec.get('redrob_signals',{}) or {}
    terms=evidence_terms(rec)
    term_txt=', '.join(terms[:3]) if terms else 'applied ML/product engineering evidence'
    la=parse_date(sig.get('last_active_date'))
    inactive=(TODAY-la).days if la else None
    resp=sig.get('recruiter_response_rate')
    notice=sig.get('notice_period_days')
    loc=p.get('location') or p.get('country') or 'location unspecified'
    effective_yoe = row.get('yoe', p.get('years_of_experience'))
    first=(f"{effective_yoe} years as {p.get('current_title')} in {p.get('current_industry')}, "
           f"with profile/career evidence for {term_txt}.")
    bits=[]
    if resp is not None: bits.append(f"{round(resp*100)}% recruiter response")
    if inactive is not None: bits.append(f"active {inactive} days ago")
    if notice is not None: bits.append(f"{notice}-day notice")
    bits.append(loc)
    second='Behavioral/logistics signals: ' + ', '.join(bits) + '.'
    return first + ' ' + second

order = list(top.index)
out = []
submission_rows=[]
for rank, cid in enumerate(order, start=1):
    r = top.loc[cid]
    rec = recs.get(cid, {})
    reasoning = make_reasoning(rec, r)
    out.append({
        'rank': rank,
        'candidate_id': cid,
        'scores': {
            'final':        round(float(r['final']), 4),
            'base_final':   round(float(r['base_final']), 4),
            'assessment_refine': round(float(r['assessment_refine']), 3),
            'sem':          round(float(r['sem_sim']), 4),
            'struct_score': round(float(r['struct_score']), 3),
            'fit':          round(float(r['fit']), 4),
            'behavioral':   round(float(r['behavioral']), 3),
            'logistics':    round(float(r['logistics']), 3),
            'tier':         int(r['tier']),
            'effective_yoe':  round(float(r['yoe']), 2),
            'profile_yoe':    (None if pd.isna(r.get('profile_yoe')) else round(float(r.get('profile_yoe')), 2)),
            'career_yoe':     (None if pd.isna(r.get('career_yoe')) else round(float(r.get('career_yoe')), 2)),
            'span_yoe':       (None if pd.isna(r.get('span_yoe')) else round(float(r.get('span_yoe')), 2)),
        },
        'reasoning':      reasoning,
        'profile':        rec.get('profile', {}),
        'career_history': rec.get('career_history', []),
        'education':      rec.get('education', []),
        'skills':         rec.get('skills', []),
        'redrob_signals': rec.get('redrob_signals', {}),
    })
    submission_rows.append({
        'candidate_id': cid,
        'rank': rank,
        'score': round(float(r['final']), 6),
        'reasoning': reasoning,
    })
with open('artifacts/top100.json', 'w') as fo:
    json.dump(out, fo, indent=2)
with open('artifacts/submission.csv', 'w', newline='') as fo:
    writer=csv.DictWriter(fo, fieldnames=['candidate_id','rank','score','reasoning'])
    writer.writeheader(); writer.writerows(submission_rows)
print('wrote artifacts/top100.json + artifacts/submission.csv with', len(out), 'candidates (rank order)')


wrote artifacts/top100.json + artifacts/submission.csv with 100 candidates (rank order)
